# LLDVE runs: corn east of the 100th meridian

Model 3 on `master_panel_corn_east100m.csv`: 31 states, 2,623 counties in the file of which 1,859 have any corn yield and 564 pass the missing-years rule. One pooled run over everything that survives the filter. With N in the hundreds the bootstrap takes a while; keep `B` low until the setup is right. Shared code is in `lldve_runs.py`.

## Imports

In [2]:
import os
import time
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import lldve_runs
from methods import LLDVE_test
import model_builder

# pick up edits to the modules without restarting the kernel
importlib.reload(model_builder)
importlib.reload(LLDVE_test)
importlib.reload(lldve_runs)

start_timer = time.time()

RecursionError: maximum recursion depth exceeded

## Settings

Everything that can be changed is here. The run tags are built as `<states>_<model>_<season>_logY_<processing>X`, the same format as the thesis results.

In [ ]:
# --- data ---
MASTER_PANEL = "master_panel_corn_east100m.csv"
START_YEAR = 1951          # nclimgrid-daily starts here, nothing to model before
END_YEAR = 2024            # must equal END_YEAR in pre_diagnostics/missing_data.py. 2024 = thesis window; 2025 is still incomplete on the yield side
MAX_MISSING_YEARS = 4      # must equal MISSING_YEAR_TOLERANCE in pre_diagnostics/missing_data.py (Zipper et al. 2016, same as thesis)

# --- model ---
SETTINGS = {
    'model_id': 'model_3',          # GDD + KDD + PREC + PREC^2 + KDD x PREC, paper Eq. 6
    'season': 'GS',                 # 1 April - 30 September
    'processing_type': 'anomaly',   # regressors as anomalies vs the county's 1981-2010 mean
    'saving_path': 'paper_main_results/corn_east100m',

    'bandwidth_method': 'manual',   # 'manual' (uses manual_h) | 'plmcv' | 'louocv' | 'aic' | 'gcv'
    # fixed bandwidth for 'manual'. The thesis PLMCV values were 0.26-0.34 (per run in lldve_run_corn); still to be revisited
    'manual_h': 0.30,
    'B': 149,                       # bootstrap replications; 149 for checking, 1499 for the final runs
    'run_bootstrap': True,
    'plot_coeffs': False,
    'plot_mean_fits': True,
    'save_draws': True,
}

## Load the panel and select counties

`select_regions` keeps the year window, drops counties without any yield, and applies the missing-years rule. The table shows every county that was dropped and why.

In [ ]:
df_all = lldve_runs.read_master_panel(MASTER_PANEL)

# the county filter was applied by pre_diagnostics/missing_data.py; stop if it used another window than this notebook
panel_filter = lldve_runs.check_panel_filter(MASTER_PANEL, START_YEAR, END_YEAR, MAX_MISSING_YEARS)

# year window, plus a re-check of the filter: with matching settings nothing is dropped here
df_model, dropped = lldve_runs.select_regions(
    df_all,
    start_year=START_YEAR,
    end_year=END_YEAR,
    max_missing_years=MAX_MISSING_YEARS,
)

print("\ncounties kept per state:")
print(df_model.groupby(df_model['fips_full'].str[:2])['fips_full'].nunique().to_string())
dropped

## Runs

One entry per run: label -> list of state codes (`None` = every state in the panel). The label becomes the first part of the output folder name.

In [ ]:
RUNS = {
    '17':       ['17'],   # Illinois, first test run (same state as the corn and soy tests)
    # 'east100m': None,     # None = every state left in df_model after select_regions, the real run
}

## Estimate

Each run writes its own folder under `saving_path`: `mTheta_hat.csv`, `vAlpha_hat.csv`, `h_optimal.txt`, fitted values and residuals, the bootstrap CI CSVs and figures, R2 and explained-variation output.

In [ ]:
results = {}
for label, states in RUNS.items():
    results[label] = lldve_runs.run_lldve(
        df_model, states, SETTINGS, states_label=label,
        run_r_squared=True, run_explained_variation=True,
    )

print(f"\nAll runs done in {(time.time() - start_timer) / 60:.1f} minutes.")

# short overview of what was estimated
summary = []
for label, res in results.items():
    summary.append({'run': label, 'N': res['N'], 'T': res['T'], 'h': res['h_optimal']})

pd.DataFrame(summary)

## Coefficient paths

In [ ]:
lldve_runs.plot_coefficient_paths(
    results,
    title="Corn east of the 100th meridian, model 3",
    save_path=os.path.join(SETTINGS['saving_path'], "coefficient_paths.png"),
)

## Counties per state after the filter

In [ ]:
kept = df_model.groupby(df_model['fips_full'].str[:2])['fips_full'].nunique().sort_values(ascending=False)
print(f"{len(kept)} states, {kept.sum()} counties")
kept.to_frame('counties')